# 🛡️ CNN Phân Loại Mã Độc — PTQ INT8 (Chi-You Style)

**Phong cách:** Post-Training Quantization theo cách Chi-You làm với LeNet-5 MNIST  
**Kiến trúc:** CNN (8×32×1 input)  
**Quantization:** PTQ — Train float32 đến convergence → Calibrate → Quantize  
**Mục tiêu:** Xuất weights INT8 thuần túy để nạp thẳng lên phần cứng (FPGA/ASIC)  

---

## 📌 Chi-You PTQ vs QAT — Tại Sao PTQ Tốt Hơn?

| Tiêu chí | PTQ (Chi-You Style) | QAT (Notebook cũ) |
|---|---|---|
| **Train float32** | ✅ Đến convergence thật sự (50 epochs) | ⚠️ 10 epochs (chưa đủ) |
| **Quantize** | ✅ SAU khi train xong | ❌ TRONG khi train (fake-quant) |
| **Calibration** | ✅ Rõ ràng, có thể điều chỉnh | ❌ Implicit, khó kiểm soát |
| **Accuracy drop** | ✅ Thường < 0.5% | ⚠️ Có thể cao hơn |
| **Debug** | ✅ Dễ — float32 và INT8 tách biệt | ❌ Khó — trộn lẫn |
| **Hardware** | ✅ Giống nhau | ✅ Giống nhau |

---

## 🔑 Ý Tưởng Cốt Lõi: Tại Sao Calibration Quan Trọng?

```
Float32 weight ∈ [-0.85, +0.85]
         ↓ scale_factor = 0.85 / 127 = 0.0067
INT8 weight   ∈ [-127, +127]

Khi nhân INT8 × INT8 → ACC INT32
Sau đó:  ACC_INT32 >> SHIFT_BITS → INT8

❓ SHIFT_BITS = bao nhiêu?
→ Calibration tìm shift_bits sao cho không bị:
   - Overflow (quá lớn, clip hết)
   - Underflow (quá nhỏ, toàn 0)
```

---

### Luồng xử lý (Chi-You style)
```
Dataset ảnh PNG
    ↓
Tiền xử lý → cache dataset.npz  
    ↓
Train float32 (50 epochs, EarlyStopping) ← ĐẦY ĐỦ
    ↓
Calibration trên validation set:
    - Chạy forward pass float32
    - Thu thập min/max activation từng layer
    - Tính scale_factor và shift_bits (per-layer)
    ↓
Quantize weights → INT8
    ↓
Verify: simulate INT8 inference trong software
    → So sánh accuracy float32 vs INT8
    ↓
Export: weights INT8 + shift_params → JSON / Excel / HEX
```

### Quy ước INT8
| Vị trí | Range | Ghi chú |
|---|---|---|
| Weights (Conv, Dense) | [-127, 127] | Signed INT8, symmetric |
| Input ảnh | [0, 127] | uint8 (0-255) >> 1 |
| Activations (ReLU) | [0, 127] | Unsigned INT8 |
| Accumulator MAC | INT32 | Tránh overflow |
| Output Dense(2) | INT32 thô | So sánh trực tiếp |

> ⚠️ Bật GPU trước khi chạy: `Runtime → Change runtime type → T4 GPU`

## 📦 1. Cài Đặt Thư Viện

In [ ]:
!pip install -q openpyxl
print('✅ Cài đặt xong!')

## ☁️ 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mount thành công!')

## ⚙️ 3. Cấu Hình Tham Số

In [ ]:
import os

# ================================================================
#  THAY ĐỔI CÁC GIÁ TRỊ NÀY TRƯỚC KHI CHẠY
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/CNX/malimg'
OUTPUT_DIR   = '/content/drive/MyDrive/CNX'
RANDOM_SEED  = 42

# ── Siêu tham số huấn luyện ──────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS_F32   = 50    # Train đầy đủ đến convergence (Chi-You style)
TEST_SPLIT   = 0.3
CALIB_SIZE   = 200   # Số ảnh dùng để calibration

# ── Tham số Quantization ─────────────────────────────────────────
Q_MAX        = 127   # INT8 symmetric: [-127, 127]

# ── Đường dẫn ────────────────────────────────────────────────────
CACHE_PATH   = os.path.join(OUTPUT_DIR, 'cache', 'dataset.npz')

print('=' * 50)
print(f'📂 Dataset     : {DATASET_PATH}')
print(f'📁 Output      : {OUTPUT_DIR}')
print(f'💾 Cache       : {CACHE_PATH}')
print(f'🎲 Seed        : {RANDOM_SEED}')
print(f'🔁 Epochs F32  : {EPOCHS_F32}')
print(f'📊 Calib size  : {CALIB_SIZE}')
print('=' * 50)

## 📚 4. Import Thư Viện

In [ ]:
import os, glob, random, time, json
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Conv2D, MaxPooling2D,
    BatchNormalization, Input
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

# Set random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Tạo thư mục cần thiết
for d in [OUTPUT_DIR,
          os.path.join(OUTPUT_DIR, 'cache'),
          os.path.join(OUTPUT_DIR, 'hardware'),
          os.path.join(OUTPUT_DIR, 'dat'),
          os.path.join(OUTPUT_DIR, 'test')]:
    os.makedirs(d, exist_ok=True)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

## 📂 5. Đọc & Tiền Xử Lý Dữ Liệu (Có Cache)

**Giống notebook cũ** — load cache nếu có, đọc ảnh nếu chưa.

**Điểm khác:** Tách thêm `X_calib` — tập dùng riêng cho calibration (không overlap test).

In [ ]:
if os.path.exists(CACHE_PATH):
    print(f'✅ Tìm thấy cache tại: {CACHE_PATH}')
    cache   = np.load(CACHE_PATH)
    X_train = cache['X_train']
    X_test  = cache['X_test']
    y_train = cache['y_train']
    y_test  = cache['y_test']
    print(f'   X_train : {X_train.shape} | y_train : {y_train.shape}')
    print(f'   X_test  : {X_test.shape}  | y_test  : {y_test.shape}')
    print('✅ Load cache hoàn tất!')
else:
    print('⚠️  Không tìm thấy cache — Đang đọc ảnh từ dataset...')
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f'❌ Không tìm thấy dataset tại: {DATASET_PATH}')

    benign_paths, malware_paths = [], []
    for fam_name in os.listdir(DATASET_PATH):
        fam_dir = os.path.join(DATASET_PATH, fam_name)
        if not os.path.isdir(fam_dir): continue
        imgs = glob.glob(os.path.join(fam_dir, '*.png'))
        if fam_name.lower() == 'benign': benign_paths.extend(imgs)
        else: malware_paths.extend(imgs)

    num_benign     = len(benign_paths)
    target_malware = num_benign * 2
    if len(malware_paths) > target_malware:
        random.seed(RANDOM_SEED)
        malware_paths = random.sample(malware_paths, target_malware)

    X, y = [], []
    for p in benign_paths:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)))
            y.append([1, 0])
    for p in malware_paths:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)))
            y.append([0, 1])

    X = np.array(X).astype('float32') / 255.0
    y = np.array(y)
    X = X.reshape(-1, 8, 32, 1)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED
    )
    np.savez_compressed(CACHE_PATH,
                        X_train=X_train, X_test=X_test,
                        y_train=y_train, y_test=y_test)
    print(f'✅ Đã lưu cache tại: {CACHE_PATH}')

# ── Tách calibration set từ train set ────────────────────────────
# QUAN TRỌNG: không dùng test set để calibrate → tránh data leakage
calib_idx = np.random.choice(len(X_train), CALIB_SIZE, replace=False)
X_calib   = X_train[calib_idx]
y_calib   = y_train[calib_idx]
print(f'\n📊 Calibration set: {X_calib.shape} ({CALIB_SIZE} samples)')
print(f'   Train: {X_train.shape} | Test: {X_test.shape}')

## 🏗️ 6. Định Nghĩa Model Float32 (Thuần Keras)

**Khác với notebook cũ:** Dùng **Keras thuần** — không có QATConv2D, không có fake-quant layers.  
Model này được train đầy đủ trước, sau đó mới quantize riêng biệt.

```
Chi-You style:
   Model Float32 ← Train bình thường, Keras standard layers
   ↓
   Calibration  ← Extract activations, tìm scale factors
   ↓
   Quantize     ← Tách biệt, không ảnh hưởng training
```

In [ ]:
def build_model_float32(input_shape=(8, 32, 1), num_classes=2):
    """
    Model thuần Keras Float32 — giống kiến trúc notebook cũ nhưng
    KHÔNG có QAT layers, KHÔNG có fake-quant, KHÔNG có STE.
    BatchNorm vẫn giữ (sẽ fold vào weights khi quantize).
    """
    inp = keras.Input(shape=input_shape, name='Input')

    # Conv Block 1
    x = Conv2D(16, (3, 3), strides=(1,1), padding='valid',
               use_bias=False, name='Conv1')(inp)
    x = BatchNormalization(name='BN1')(x)
    x = layers.ReLU(name='ReLU1')(x)
    x = MaxPooling2D(pool_size=(2,2), name='Pool1')(x)

    # Conv Block 2
    x = Conv2D(32, (3, 3), strides=(1,1), padding='valid',
               use_bias=False, name='Conv2')(x)
    x = BatchNormalization(name='BN2')(x)
    x = layers.ReLU(name='ReLU2')(x)

    # Dense Block
    x = Flatten(name='Flatten')(x)
    x = Dense(48, use_bias=False, name='Dense1')(x)
    x = BatchNormalization(name='BN3')(x)
    x = layers.ReLU(name='ReLU3')(x)

    # Output (không có activation — raw logits)
    out = Dense(num_classes, use_bias=True, name='Dense2')(x)

    return keras.Model(inputs=inp, outputs=out, name='CNN_PTQ_Float32')


model_f32 = build_model_float32()
model_f32.compile(
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy']
)
model_f32.summary()
print(f'\nTotal params: {model_f32.count_params():,}')

## 🏋️ 7. Train Float32 (Đến Convergence)

**Điểm mấu chốt của Chi-You PTQ:**  
Train thật đầy đủ ở float32. Model phải đạt accuracy tối đa trước khi quantize.  
Dùng `EarlyStopping` + `ReduceLROnPlateau` để hội tụ tốt.

```
QAT (cũ): train 10 epochs float32 → 10 epochs QAT
           Model chưa convergence đã quantize

PTQ (Chi-You): train 50 epochs → EarlyStopping
               Model đã convergence hoàn toàn
               → Sau đó mới quantize
```

In [ ]:
print('=' * 55)
print('TRAIN FLOAT32 (Chi-You PTQ style)')
print('=' * 55)

f32_ckpt_path = os.path.join(OUTPUT_DIR, 'checkpoint_ptq_f32.weights.h5')

callbacks = [
    # Dừng sớm nếu val_accuracy không tăng sau 10 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    # Giảm LR khi loss plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1
    ),
    # Lưu checkpoint tốt nhất
    keras.callbacks.ModelCheckpoint(
        f32_ckpt_path, monitor='val_accuracy',
        save_best_only=True, save_weights_only=True, verbose=0
    ),
]

tic = time.time()
history = model_f32.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_F32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)
toc = time.time()

# Load best weights
model_f32.load_weights(f32_ckpt_path)

# Đánh giá
loss_f32, acc_f32 = model_f32.evaluate(X_test, y_test, verbose=0)
print(f'\n⏱️  Thời gian train: {toc-tic:.1f}s')
print(f'✅ Float32 Test Accuracy : {acc_f32*100:.2f}%')
print(f'✅ Float32 Test Loss     : {loss_f32:.4f}')
print(f'💾 Checkpoint lưu tại   : {f32_ckpt_path}')

## 📈 8. Biểu Đồ Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)

plt.suptitle('Float32 Training (Chi-You PTQ Style)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve_ptq.png'), dpi=150)
plt.show()
print('✅ Biểu đồ lưu tại:', os.path.join(OUTPUT_DIR, 'training_curve_ptq.png'))

## 🎯 9. Calibration — Tìm Scale Factors

**Đây là bước quan trọng nhất của PTQ (Chi-You style)!**

### Tại sao cần calibration?

```
Khi nhân INT8 × INT8:
   max(INT8) × max(INT8) = 127 × 127 = 16,129
   Với kernel 3×3×16 channels: 16,129 × 9 × 16 = 2,322,576 (vẫn trong INT32)

Sau khi MAC xong, cần shift phải để ép về INT8:
   acc_int32 >> SHIFT_BITS → INT8

SHIFT_BITS = ? ← Calibration tìm giá trị này!
```

### Cách tính SHIFT_BITS:

```
scale_input  = 1/127          (pixel uint8 >> 1 → [0,127])
scale_weight = max(|w|) / 127 (weight đã quantize, per-layer)
scale_output = max(|act|) / 127 (activation range từ calibration)

scale_acc    = scale_input × scale_weight
shift_float  = log2(scale_acc / scale_output)
SHIFT_BITS   = round(shift_float)
```

### Ví dụ cụ thể:

```
scale_input  = 1/127  ≈ 0.00787
scale_w_conv1 = 0.0067  (max weight conv1 / 127)
scale_acc_conv1 = 0.00787 × 0.0067 ≈ 5.27e-5

Từ calibration: max activation conv1 = 8.5
scale_output_conv1 = 8.5 / 127 ≈ 0.0669

shift_float = log2(0.0669 / 5.27e-5) = log2(1270) ≈ 10.3
SHIFT_BITS  = 10

→ Sau MAC: acc_int32 >> 10 → INT8  ← Hardware chỉ cần 1 lệnh
```

In [ ]:
# ================================================================
# CELL 9 — CALIBRATION (Chi-You PTQ Style)
# ================================================================

print('=' * 55)
print('CALIBRATION — Tìm scale factors từng layer')
print('=' * 55)

# ── Bước 1: Lấy activations từng layer trên calibration set ─────
# Tạo model phụ để extract output từng layer
layer_names = ['ReLU1', 'ReLU2', 'ReLU3']  # Sau mỗi ReLU
activation_extractor = keras.Model(
    inputs=model_f32.input,
    outputs=[model_f32.get_layer(n).output for n in layer_names]
)

print(f'\n📊 Chạy forward pass trên {CALIB_SIZE} ảnh calibration...')
acts = activation_extractor.predict(X_calib, batch_size=32, verbose=0)

# ── Bước 2: Tính max activation từng layer ───────────────────────
calib_stats = {}
print('\n📐 Thống kê activation range (sau ReLU):')
print(f'   {"Layer":<10} {"Min":>8} {"Max":>8} {"P99.9":>8} {"Scale":>10} {"Shift":>6}')
print('   ' + '-'*55)

for i, name in enumerate(layer_names):
    act_flat = acts[i].flatten()

    act_min  = float(act_flat.min())
    act_max  = float(act_flat.max())

    # Dùng percentile 99.9 thay vì max để loại outliers
    # → Đây là điểm quan trọng: max tuyệt đối dễ bị outlier làm hỏng
    act_p999 = float(np.percentile(act_flat, 99.9))

    # Scale output: ánh xạ [0, act_p999] → [0, 127]
    scale_out = act_p999 / Q_MAX
    scale_out = max(scale_out, 1e-6)  # tránh chia 0

    # Tính shift bits:
    # acc_int32 tích lũy trên scale_input × scale_weight
    # Để ép về INT8 trên scale_out:
    #   (scale_input × scale_weight) / scale_out = 1 / 2^shift
    #   → shift = log2(scale_out / (scale_input × scale_weight))
    # Giá trị scale_input = 1/127 (input được quantize [0,127])
    scale_input = 1.0 / Q_MAX

    # scale_weight từ layer tương ứng (tính sau khi fold BN)
    # Tạm thời dùng giá trị mặc định; sẽ tính chính xác sau fold
    calib_stats[name] = {
        'min'      : act_min,
        'max'      : act_max,
        'p999'     : act_p999,
        'scale_out': scale_out,
    }
    print(f'   {name:<10} {act_min:>8.3f} {act_max:>8.3f} {act_p999:>8.3f} {scale_out:>10.6f}')

print('\n✅ Calibration hoàn tất!')
print('   → Các giá trị p99.9 sẽ dùng để tính SHIFT_BITS từng layer')

## 🔧 10. Fold BatchNorm + Quantize Weights

**Fold BatchNorm (BN Folding):**  
BatchNorm có 4 tham số: γ, β, mean, var.  
Khi inference, BN thực hiện:
```
output = γ × (conv_out - mean) / sqrt(var + ε) + β
```
Ta có thể gộp BN vào Conv weight:
```
w_fold = w × γ / sqrt(var + ε)        ← hấp thụ scale
b_fold = (-mean × γ / sqrt(var + ε)) + β  ← hấp thụ shift
```
Kết quả: Hardware không cần tính BN nữa, chỉ cần Conv + bias!

**Sau fold, quantize về INT8:**
```
scale_w = max(|w_fold|) / 127
w_int8  = round(w_fold / scale_w)  ∈ [-127, 127]
```

In [ ]:
# ================================================================
# CELL 10 — FOLD BN + QUANTIZE WEIGHTS
# ================================================================

print('=' * 55)
print('FOLD BN + QUANTIZE WEIGHTS → INT8')
print('=' * 55)

# ── Helper functions ──────────────────────────────────────────────

def fold_bn_conv(conv_name, bn_name):
    """Fold BatchNorm vào Conv weights."""
    w    = model_f32.get_layer(conv_name).kernel.numpy()  # (kH, kW, Cin, Cout)
    bn   = model_f32.get_layer(bn_name)
    γ    = bn.gamma.numpy()
    β    = bn.beta.numpy()
    μ    = bn.moving_mean.numpy()
    σ2   = bn.moving_variance.numpy()
    ε    = 1e-3

    scale  = γ / np.sqrt(σ2 + ε)                          # (Cout,)
    w_fold = w * scale[np.newaxis, np.newaxis, np.newaxis, :]  # broadcast
    b_fold = (-μ * scale) + β
    return w_fold, b_fold


def fold_bn_dense(dense_name, bn_name):
    """Fold BatchNorm vào Dense weights."""
    w    = model_f32.get_layer(dense_name).kernel.numpy()  # (Cin, Cout)
    bn   = model_f32.get_layer(bn_name)
    γ    = bn.gamma.numpy()
    β    = bn.beta.numpy()
    μ    = bn.moving_mean.numpy()
    σ2   = bn.moving_variance.numpy()
    ε    = 1e-3

    scale  = γ / np.sqrt(σ2 + ε)
    w_fold = w * scale[np.newaxis, :]  # (Cin, Cout)
    b_fold = (-μ * scale) + β
    return w_fold, b_fold


def quantize_weight_perchannel(w_fold, q_max=127):
    """
    Quantize weight per-output-channel (symmetric INT8).
    Per-channel: mỗi output channel có scale riêng
    → Tốt hơn per-tensor vì tránh channel lớn dominate scale
    """
    # axis cuối = output channel (Cout)
    reduce_axes = tuple(range(len(w_fold.shape) - 1))
    scale_w = np.max(np.abs(w_fold), axis=reduce_axes, keepdims=True)  # (1,1,...,Cout)
    scale_w = np.maximum(scale_w, 1e-6)
    w_int8  = np.round(w_fold / scale_w * q_max)
    w_int8  = np.clip(w_int8, -q_max, q_max).astype(np.int8)
    return w_int8, scale_w.squeeze()  # scale_w: (Cout,)


def quantize_bias(b_fold, scale_input, scale_w, q_max=127):
    """
    Quantize bias theo scale của accumulator:
    scale_bias = scale_input × scale_w
    bias_int32 = round(b_fold / scale_bias)

    Giữ bias ở INT32 để tránh mất precision.
    """
    scale_bias = scale_input * scale_w  # per-channel
    bias_int32 = np.round(b_fold / scale_bias).astype(np.int32)
    return bias_int32


def compute_shift_bits(scale_input, scale_w, scale_output):
    """
    Tính SHIFT_BITS từ scale factors:

    Lý thuyết:
      ACC_int32 = Σ (input_int8 × weight_int8)
      Giá trị thực của ACC: acc_float = acc_int32 × scale_input × scale_w

      Ta muốn: acc_float = out_int8 × scale_output
      → out_int8 = acc_int32 × (scale_input × scale_w) / scale_output
                 = acc_int32 × multiplier

    Nếu multiplier ≈ 1/2^shift:
      out_int8 = acc_int32 >> shift  ← Hardware đơn giản!

    shift = log2(scale_output / (scale_input × scale_w))
    """
    # Per-channel: scale_w là vector (Cout,)
    multiplier  = (scale_input * scale_w) / scale_output
    shift_float = -np.log2(multiplier)  # shift > 0 → dịch phải
    shift_bits  = int(np.round(np.mean(shift_float)))  # dùng mean nếu per-channel
    shift_bits  = max(0, min(shift_bits, 31))           # clamp [0, 31]
    return shift_bits


# ── Fold + Quantize từng layer ────────────────────────────────────
scale_input = 1.0 / Q_MAX  # Input: pixel [0,255] → [0,127], scale = 1/127

# === CONV1 ===
w_c1f, b_c1f = fold_bn_conv('Conv1', 'BN1')
w_c1_int8, scale_w_c1 = quantize_weight_perchannel(w_c1f)
# scale_out từ calibration (ReLU1)
scale_out_c1 = calib_stats['ReLU1']['scale_out']
shift_c1     = compute_shift_bits(scale_input, scale_w_c1, scale_out_c1)
b_c1_int32   = quantize_bias(b_c1f, scale_input, scale_w_c1)
# Transpose sang hardware shape: (kH, kW, Cin, Cout) → (Cout, Cin, kH, kW)
w_c1_hw      = w_c1_int8.transpose(3, 2, 0, 1)  # (16, 1, 3, 3)

print(f'Conv1  : W{w_c1_hw.shape}  range=[{w_c1_int8.min()},{w_c1_int8.max()}]')
print(f'         scale_w max={scale_w_c1.max():.4f}  scale_out={scale_out_c1:.4f}  SHIFT={shift_c1}')

# === CONV2 ===
# Input của Conv2 là output của ReLU1 với scale scale_out_c1
w_c2f, b_c2f = fold_bn_conv('Conv2', 'BN2')
w_c2_int8, scale_w_c2 = quantize_weight_perchannel(w_c2f)
scale_out_c2 = calib_stats['ReLU2']['scale_out']
shift_c2     = compute_shift_bits(scale_out_c1, scale_w_c2, scale_out_c2)
b_c2_int32   = quantize_bias(b_c2f, scale_out_c1, scale_w_c2)
w_c2_hw      = w_c2_int8.transpose(3, 2, 0, 1)  # (32, 16, 3, 3)

print(f'\nConv2  : W{w_c2_hw.shape}  range=[{w_c2_int8.min()},{w_c2_int8.max()}]')
print(f'         scale_w max={scale_w_c2.max():.4f}  scale_out={scale_out_c2:.4f}  SHIFT={shift_c2}')

# === DENSE1 ===
w_d1f, b_d1f = fold_bn_dense('Dense1', 'BN3')
w_d1_int8, scale_w_d1 = quantize_weight_perchannel(w_d1f)
scale_out_d1 = calib_stats['ReLU3']['scale_out']
shift_d1     = compute_shift_bits(scale_out_c2, scale_w_d1, scale_out_d1)
b_d1_int32   = quantize_bias(b_d1f, scale_out_c2, scale_w_d1)

print(f'\nDense1 : W{w_d1_int8.shape}  range=[{w_d1_int8.min()},{w_d1_int8.max()}]')
print(f'         scale_w max={scale_w_d1.max():.4f}  scale_out={scale_out_d1:.4f}  SHIFT={shift_d1}')

# === DENSE2 (output layer) ===
# Không có BN, không có ReLU → giữ INT32 thô
w_d2f   = model_f32.get_layer('Dense2').kernel.numpy()   # (48, 2)
b_d2f   = model_f32.get_layer('Dense2').bias.numpy()     # (2,)
scale_w_d2 = np.maximum(np.max(np.abs(w_d2f)), 1e-6)
w_d2_int8  = np.clip(np.round(w_d2f / scale_w_d2 * Q_MAX), -Q_MAX, Q_MAX).astype(np.int8)
b_d2_int32 = np.round(b_d2f / (scale_out_d1 * scale_w_d2 / Q_MAX)).astype(np.int32)
# Dense2 output giữ INT32 → không cần shift_d2, không có scale_out

print(f'\nDense2 : W{w_d2_int8.shape}  range=[{w_d2_int8.min()},{w_d2_int8.max()}]')
print(f'         scale_w={scale_w_d2:.4f}  (output giữ INT32, không shift)')

# ── Lưu shift_params ─────────────────────────────────────────────
shift_params = {
    'Conv1'  : shift_c1,
    'Conv2'  : shift_c2,
    'Dense1' : shift_d1,
    'Dense2' : None,    # Giữ INT32
    'scale_input'   : float(scale_input),
    'scale_out_c1'  : float(scale_out_c1),
    'scale_out_c2'  : float(scale_out_c2),
    'scale_out_d1'  : float(scale_out_d1),
}
shift_path = os.path.join(OUTPUT_DIR, 'hardware', 'shift_params.json')
with open(shift_path, 'w') as f:
    json.dump(shift_params, f, indent=2)
print(f'\n✅ Shift params: {shift_params}')
print(f'💾 Lưu tại: {shift_path}')

## ✅ 11. Verify — Simulate INT8 Inference trong Software

**Bước quan trọng nhất để đảm bảo hardware đúng:**  
Mô phỏng lại toàn bộ INT8 inference bằng numpy, không dùng Keras.  
So sánh kết quả với float32 model.

```
Input (float32) → Quantize → INT8
   ↓
Conv1 INT8 MAC → ACC INT32 → >> shift_c1 → clip → INT8
   ↓ ReLU (clip negative → 0)
MaxPool 2×2
   ↓
Conv2 INT8 MAC → ACC INT32 → >> shift_c2 → clip → INT8
   ↓ ReLU
Flatten → 416 INT8
   ↓
Dense1 INT8 MAC → ACC INT32 → >> shift_d1 → clip → INT8
   ↓ ReLU
Dense2 INT8 MAC → ACC INT32 (GIỮ NGUYÊN)
   ↓
if acc[0] > acc[1]: BENIGN else MALWARE
```

In [ ]:
# ================================================================
# CELL 11 — SIMULATE INT8 INFERENCE (numpy, giống hardware)
# ================================================================

def conv2d_int8(x_int8, w_int8, b_int32, shift_bits,
                stride=1, relu=True):
    """
    INT8 Convolution simulation:
    - x_int8  : (H, W, Cin)    — INT8 input
    - w_int8  : (Cout, Cin, kH, kW) — hardware shape → transpose về (kH, kW, Cin, Cout)
    - b_int32 : (Cout,)         — INT32 bias
    - shift_bits: scalar
    """
    H, W, Cin = x_int8.shape
    w_tf = w_int8.transpose(2, 3, 1, 0).astype(np.int32)  # → (kH, kW, Cin, Cout)
    kH, kW = w_tf.shape[0], w_tf.shape[1]
    Cout = w_tf.shape[3]

    # Output size (valid padding)
    Ho = (H - kH) // stride + 1
    Wo = (W - kW) // stride + 1
    out = np.zeros((Ho, Wo, Cout), dtype=np.int32)

    # MAC (mô phỏng hardware, dùng INT32 accumulator)
    for i in range(Ho):
        for j in range(Wo):
            patch = x_int8[i*stride:i*stride+kH,
                           j*stride:j*stride+kW, :].astype(np.int32)  # (kH, kW, Cin)
            # (kH, kW, Cin) × (kH, kW, Cin, Cout) → sum → (Cout,)
            out[i, j, :] = np.einsum('hwc,hwco->o', patch, w_tf) + b_int32

    # Shift phải (thay cho chia) → ép về INT8
    out_shifted = out >> shift_bits
    if relu:
        out_shifted = np.maximum(out_shifted, 0)    # ReLU
    out_int8 = np.clip(out_shifted, -127, 127).astype(np.int8)
    return out_int8


def maxpool2d(x_int8, pool_size=2):
    """MaxPooling 2×2 — không cần tham số."""
    H, W, C = x_int8.shape
    Ho, Wo  = H // pool_size, W // pool_size
    out = np.zeros((Ho, Wo, C), dtype=np.int8)
    for i in range(Ho):
        for j in range(Wo):
            out[i, j, :] = x_int8[i*pool_size:(i+1)*pool_size,
                                   j*pool_size:(j+1)*pool_size, :].max(axis=(0,1))
    return out


def dense_int8(x_int8, w_int8, b_int32, shift_bits, relu=True):
    """
    INT8 Dense simulation:
    - x_int8 : (N,)       — INT8 input vector
    - w_int8 : (Cin, Cout) — weight
    - b_int32: (Cout,)
    """
    acc = x_int8.astype(np.int32) @ w_int8.astype(np.int32) + b_int32
    if shift_bits is not None:
        acc = acc >> shift_bits
    if relu:
        acc = np.maximum(acc, 0)
        acc = np.clip(acc, 0, 127).astype(np.int8)
    return acc  # INT32 nếu không relu (output layer)


def simulate_int8_inference(x_float32):
    """
    Toàn bộ INT8 inference pipeline — giống hardware.
    Input: (8, 32, 1) float32 ∈ [0, 1]
    Output: (2,) INT32 logits
    """
    # Bước 1: Quantize input
    x = np.round(x_float32 * Q_MAX).astype(np.int8)  # [0, 127]

    # Bước 2: Conv1
    x = conv2d_int8(x.squeeze(-1)[:,:,np.newaxis]  # (8,32,1)
                    if len(x.shape)==3 else x,
                    w_c1_hw, b_c1_int32.astype(np.int32), shift_c1, relu=True)
    # x: (6, 30, 16)

    # Bước 3: MaxPool1
    x = maxpool2d(x)  # (3, 15, 16)

    # Bước 4: Conv2
    x = conv2d_int8(x, w_c2_hw, b_c2_int32.astype(np.int32), shift_c2, relu=True)
    # x: (1, 13, 32)

    # Bước 5: Flatten
    x = x.flatten()  # 416

    # Bước 6: Dense1
    x = dense_int8(x, w_d1_int8, b_d1_int32.astype(np.int32), shift_d1, relu=True)
    # x: (48,) INT8

    # Bước 7: Dense2 (output — giữ INT32)
    x = dense_int8(x, w_d2_int8, b_d2_int32.astype(np.int32), shift_bits=None, relu=False)
    # x: (2,) INT32

    return x


# ── Verify trên test set ──────────────────────────────────────────
print('=' * 55)
print('VERIFY: Float32 vs INT8 Simulation')
print('=' * 55)

N_verify = min(200, len(X_test))
correct_int8  = 0
correct_f32   = 0
agreed        = 0

for i in range(N_verify):
    x_sample = X_test[i]          # (8, 32, 1)
    y_true   = np.argmax(y_test[i])

    # Float32 prediction
    logits_f32 = model_f32.predict(x_sample[np.newaxis], verbose=0)[0]
    pred_f32   = np.argmax(logits_f32)

    # INT8 simulation
    logits_int8 = simulate_int8_inference(x_sample)
    pred_int8   = np.argmax(logits_int8)

    if pred_f32  == y_true: correct_f32  += 1
    if pred_int8 == y_true: correct_int8 += 1
    if pred_f32  == pred_int8: agreed    += 1

acc_f32_verify  = correct_f32  / N_verify * 100
acc_int8_verify = correct_int8 / N_verify * 100
agreement       = agreed / N_verify * 100

print(f'\nVerify trên {N_verify} mẫu:')
print(f'  Float32 Accuracy  : {acc_f32_verify:.1f}%')
print(f'  INT8    Accuracy  : {acc_int8_verify:.1f}%')
print(f'  Agreement Rate    : {agreement:.1f}%')
print(f'  Accuracy Drop     : {acc_f32_verify - acc_int8_verify:.2f}%')

if acc_f32_verify - acc_int8_verify < 1.0:
    print('\n✅ Accuracy drop < 1% — Quantization thành công!')
else:
    print('\n⚠️  Accuracy drop > 1% — Kiểm tra lại shift_bits hoặc calibration!')

## 📊 12. Đánh Giá Toàn Bộ Test Set

In [ ]:
print('Đánh giá Float32 model trên toàn bộ test set...')
loss_f32, acc_f32 = model_f32.evaluate(X_test, y_test, verbose=0)

# Dự đoán
y_pred_f32_probs = model_f32.predict(X_test, verbose=0)
y_pred_f32 = np.argmax(y_pred_f32_probs, axis=1)
y_true     = np.argmax(y_test, axis=1)

target_names = ['Benign', 'Malware']

print(f'\n✅ Float32 Test Accuracy: {acc_f32*100:.2f}%')
print('\n' + classification_report(y_true, y_pred_f32, target_names=target_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_f32)
cm_norm = np.around(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis], 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm,      annot=True, fmt='d',    cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=ax1)
ax1.set_title('Float32 Confusion Matrix (Raw)')
sns.heatmap(cm_norm, annot=True, fmt='.2f',  cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax2)
ax2.set_title('Float32 Confusion Matrix (Normalized)')
plt.suptitle(f'PTQ Style — Float32 Accuracy: {acc_f32*100:.2f}%')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_ptq.png'), dpi=150)
plt.show()

## 💾 13. Export Weights INT8 + HEX cho Hardware

Xuất đầy đủ weights INT8 theo format hardware:
- `bram_w1.hex` — Conv1 weights + bias
- `bram_w2.hex` — Conv2 weights + bias  
- `bram_w3.hex` — Dense1 weights + bias
- `bram_w4.hex` — Dense2 weights + bias
- `weights_int8.json` — Toàn bộ weights + shift_params
- `weights_int8.xlsx` — Dạng bảng để debug
- `shift_params.json` — SHIFT_BITS từng layer

In [ ]:
# ================================================================
# CELL 13 — EXPORT WEIGHTS INT8 + HEX
# ================================================================

hw_dir = os.path.join(OUTPUT_DIR, 'hardware')

# ── Helper: export weights → HEX file ────────────────────────────
def export_hex(weights_int8, bias_int8, filename, words_per_line=4):
    """
    Xuất weights INT8 dạng hex, 4 bytes/word (INT32).
    Format: pack 4 INT8 → 1 INT32 word → hex string

    Hardware đọc theo word (32-bit), mỗi word = 4 weights.
    """
    flat = np.concatenate([weights_int8.flatten(),
                           bias_int8.flatten()]).astype(np.int8)
    # Padding đến bội số 4
    pad = (4 - len(flat) % 4) % 4
    flat = np.concatenate([flat, np.zeros(pad, dtype=np.int8)])

    lines = []
    for i in range(0, len(flat), 4):
        b0, b1, b2, b3 = flat[i:i+4].view(np.uint8)
        word = (int(b3) << 24) | (int(b2) << 16) | (int(b1) << 8) | int(b0)
        lines.append(f'{word:08x}')

    path = os.path.join(hw_dir, filename)
    with open(path, 'w') as f:
        f.write('\n'.join(lines))
    return path, len(lines)


# Convert bias sang INT8 để xuất hex (giá trị đã scale)
def bias_to_int8(b_int32, q_max=127):
    return np.clip(b_int32, -q_max, q_max).astype(np.int8)


# ── Export HEX ────────────────────────────────────────────────────
p1, n1 = export_hex(w_c1_hw,   bias_to_int8(b_c1_int32),  'bram_w1.hex')
p2, n2 = export_hex(w_c2_hw,   bias_to_int8(b_c2_int32),  'bram_w2.hex')
p3, n3 = export_hex(w_d1_int8, bias_to_int8(b_d1_int32),  'bram_w3.hex')
p4, n4 = export_hex(w_d2_int8, bias_to_int8(b_d2_int32),  'bram_w4.hex')

print('HEX files:')
for p, n, desc in [(p1,n1,'Conv1'), (p2,n2,'Conv2'), (p3,n3,'Dense1'), (p4,n4,'Dense2')]:
    size_kb = os.path.getsize(p) / 1024
    print(f'  ✅ {os.path.basename(p):<18s} [{size_kb:>6.1f} KB]  {n} words — {desc}')

# ── Export JSON ───────────────────────────────────────────────────
weights_dict = {
    'method'       : 'PTQ (Chi-You style)',
    'quantization' : 'INT8 symmetric per-channel',
    'shift_params' : shift_params,
    'Conv1'  : {'weights': w_c1_hw.tolist(),   'bias': b_c1_int32.tolist(),
                'shape_hw': list(w_c1_hw.shape), 'shift': shift_c1},
    'Conv2'  : {'weights': w_c2_hw.tolist(),   'bias': b_c2_int32.tolist(),
                'shape_hw': list(w_c2_hw.shape), 'shift': shift_c2},
    'Dense1' : {'weights': w_d1_int8.tolist(), 'bias': b_d1_int32.tolist(),
                'shape_hw': list(w_d1_int8.shape), 'shift': shift_d1},
    'Dense2' : {'weights': w_d2_int8.tolist(), 'bias': b_d2_int32.tolist(),
                'shape_hw': list(w_d2_int8.shape), 'shift': None},
}
json_path = os.path.join(hw_dir, 'weights_int8.json')
with open(json_path, 'w') as f:
    json.dump(weights_dict, f, indent=2)
print(f'\n✅ weights_int8.json : {json_path}')

# ── Export Excel ──────────────────────────────────────────────────
excel_path = os.path.join(hw_dir, 'weights_int8_ptq.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Sheet 1: Shift params
    df_shift = pd.DataFrame([
        {'Layer': k, 'Shift_Bits': v}
        for k, v in shift_params.items()
        if isinstance(v, (int, float)) or v is None
    ])
    df_shift.to_excel(writer, sheet_name='Shift_Params', index=False)

    # Sheet 2: Conv1 weights
    pd.DataFrame(w_c1_hw.reshape(w_c1_hw.shape[0], -1)).to_excel(
        writer, sheet_name='Conv1_W', index=True)

    # Sheet 3: Conv2 weights
    pd.DataFrame(w_c2_hw.reshape(w_c2_hw.shape[0], -1)).to_excel(
        writer, sheet_name='Conv2_W', index=True)

    # Sheet 4: Dense1
    pd.DataFrame(w_d1_int8).to_excel(writer, sheet_name='Dense1_W', index=True)

    # Sheet 5: Dense2
    pd.DataFrame(w_d2_int8).to_excel(writer, sheet_name='Dense2_W', index=True)

print(f'✅ weights_int8_ptq.xlsx : {excel_path}')

## 🧪 14. Test 5 Ảnh Mẫu (Export HEX input cho Hardware)

In [ ]:
# ================================================================
# CELL 14 — TEST 5 ẢNH MẪU + EXPORT INPUT HEX
# ================================================================

test_dir = os.path.join(OUTPUT_DIR, 'test')
label_names = ['Benign', 'Malware']

# Chọn 5 ảnh: 2 benign + 3 malware
benign_idx  = np.where(y_test[:, 0] == 1)[0][:2]
malware_idx = np.where(y_test[:, 1] == 1)[0][:3]
test_indices = np.concatenate([benign_idx, malware_idx])

print('=' * 65)
print('TEST 5 ẢNH MẪU (Chi-You PTQ INT8)')
print('=' * 65)

results = []
for case_no, idx in enumerate(test_indices):
    x      = X_test[idx]              # (8, 32, 1) float32
    y_true = np.argmax(y_test[idx])

    # INT8 inference
    logits = simulate_int8_inference(x)
    pred   = np.argmax(logits)
    correct = '✅ CORRECT' if pred == y_true else '❌ WRONG'

    print(f'\n[Case {case_no+1}] idx={idx} | True={label_names[y_true]:<7} '
          f'| Pred={label_names[pred]:<7} | Score=[{logits[0]:>7},{logits[1]:>7}] | {correct}')

    # Export input hex: pixels uint8, 4 pixels/word
    img_uint8 = np.round(x.squeeze() * 255).astype(np.uint8)  # (8, 32)
    flat_pixels = img_uint8.flatten()  # 256 bytes
    # Pack 4 pixels per word
    hex_lines = []
    for i in range(0, len(flat_pixels), 4):
        p = flat_pixels[i:i+4]
        word = (int(p[3]) << 24) | (int(p[2]) << 16) | (int(p[1]) << 8) | int(p[0])
        hex_lines.append(f'{word:08x}')

    hex_path = os.path.join(test_dir, f'test{case_no+1}_in32.hex')
    with open(hex_path, 'w') as f:
        f.write('\n'.join(hex_lines))

    results.append({
        'Case': case_no+1, 'Index': idx,
        'True': label_names[y_true], 'Pred': label_names[pred],
        'Score_0': int(logits[0]), 'Score_1': int(logits[1]),
        'Correct': '✓' if pred == y_true else '✗',
    })

n_correct = sum(1 for r in results if r['Correct'] == '✓')
print(f'\n✅ Kết quả: {n_correct}/5 đúng')
print(f'📁 HEX files tại: {test_dir}')

## ✅ 15. Tổng Kết Output

### Hardware Inference Flow (PTQ Chi-You Style)

```
Bước 1 — Quantize input:
  pixel_int8 = pixel_uint8 >> 1   (0..127)
  → hoặc: pixel_int8 = round(pixel_float × 127)

Bước 2 — Conv1 (BRAM_W1, shift=shift_c1):
  acc_int32 = Σ(input_int8 × weight_int8) + bias_int32
  out_int8  = clip(acc_int32 >> SHIFT_C1, 0, 127)  ← ReLU

Bước 3 — MaxPool 2×2

Bước 4 — Conv2 (BRAM_W2, shift=shift_c2):
  acc_int32 = Σ(input_int8 × weight_int8) + bias_int32
  out_int8  = clip(acc_int32 >> SHIFT_C2, 0, 127)  ← ReLU

Bước 5 — Flatten → 416 INT8

Bước 6 — Dense1 (BRAM_W3, shift=shift_d1):
  acc_int32 = Σ(input_int8 × weight_int8) + bias_int32
  out_int8  = clip(acc_int32 >> SHIFT_D1, 0, 127)  ← ReLU

Bước 7 — Dense2 (BRAM_W4, KHÔNG shift):
  acc_int32 = Σ(input_int8 × weight_int8) + bias_int32
  GIỮ NGUYÊN INT32

Bước 8 — Quyết định (không cần softmax):
  if acc[0] > acc[1]: → BENIGN
  if acc[1] > acc[0]: → MALWARE
```

### Khác biệt so với notebook QAT cũ:
| | PTQ (notebook này) | QAT (notebook cũ) |
|---|---|---|
| **Shift bits** | Tính từ calibration (**per-layer, có cơ sở**) | Cố định = 7 (thủ công) |
| **Scale factors** | Có lưu rõ ràng | Implicit trong fake-quant |
| **Bias** | INT32 (chính xác hơn) | INT8 (mất precision) |
| **Verify** | Simulate numpy INT8 giống hardware | Dùng Keras float32 |
| **Debug** | Dễ (float32 và INT8 tách biệt) | Khó (trộn lẫn) |

In [ ]:
print('\n' + '=' * 65)
print('TỔNG KẾT OUTPUT — PTQ Chi-You Style')
print('=' * 65)

file_desc = {
    'hardware/bram_w1.hex'          : f'BRAM_W1 — Conv1  weights+bias  (shift={shift_c1})',
    'hardware/bram_w2.hex'          : f'BRAM_W2 — Conv2  weights+bias  (shift={shift_c2})',
    'hardware/bram_w3.hex'          : f'BRAM_W3 — Dense1 weights+bias  (shift={shift_d1})',
    'hardware/bram_w4.hex'          : 'BRAM_W4 — Dense2 weights+bias  (no shift)',
    'hardware/weights_int8.json'    : 'Weights INT8 + shift_params',
    'hardware/weights_int8_ptq.xlsx': 'Weights + Shift Params (Excel)',
    'hardware/shift_params.json'    : 'SHIFT_BITS từng layer (từ calibration)',
    'cache/dataset.npz'             : 'Cache dataset',
    'checkpoint_ptq_f32.weights.h5' : 'Float32 model checkpoint (best)',
}

for fname, desc in file_desc.items():
    full   = os.path.join(OUTPUT_DIR, fname)
    status = '✅' if os.path.exists(full) else '❌'
    size   = f'{os.path.getsize(full)/1024:.1f} KB' if os.path.exists(full) else 'missing'
    print(f'{status}  {fname:<44s} [{size:>8s}]  {desc}')

print('\n' + '=' * 65)
print('SHIFT BITS TỪNG LAYER (từ Calibration)')
print('=' * 65)
for k, v in shift_params.items():
    if isinstance(v, int):
        print(f'  {k:<10}: SHIFT = {v}')
    elif v is None:
        print(f'  {k:<10}: Giữ INT32 (không shift)')